In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()



In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create binary label column
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# clean labels and create binary label column
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)

# Create Objects 

In [5]:
bert_binary_model = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9639.65it/s]


Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3019.30it/s]


Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22611.71it/s]


In [6]:
bert_binary_ensemble = Ensemble(
    model_collections=[bert_binary_model]
)

# Simple Majority Rules

In [ ]:
pred_train = bert_binary_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...


KeyboardInterrupt: 

In [7]:
pred_train = bert_binary_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_train)

Predicting with SimpleMajority...


KeyboardInterrupt: 